# ENARES 2024 - Stage 1 Raw Profiling

**Notebook 05:** `05_ENARES_2024_STAGE1_perfilamiento.ipynb`

This notebook generates a raw exploratory profiling report for CRS04 using `ydata-profiling`.

## Scope

This belongs to **Stage 1 - Data Ingestion**. It does **not** clean, recode, merge, model, or infer.  
It only profiles the raw CRS04 `.sav` files already ingested from the official INEI SPSS ZIP packages.

## Privacy rule

The generated HTML report may contain distributions, rare values, metadata, or sensitive raw-data patterns.  
Therefore, the HTML output must stay in Google Drive and must **not** be pushed to GitHub.


In [1]:
# ============================================================
# 1. INSTALL DEPENDENCIES
# ============================================================

!pip install -q ydata-profiling pyreadstat pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.8/400.8 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 57.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 679.7/679.7 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 5.8 MB/s eta 0:00:00


In [2]:
# ============================================================
# 2. IMPORTS AND DRIVE SETUP
# ============================================================

import os
import glob
import json
import pandas as pd
import pyreadstat
from datetime import datetime
from ydata_profiling import ProfileReport

from google.colab import drive
drive.mount('/content/drive')


/tmp/ipykernel_503/115527368.py:11: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


Mounted at /content/drive


In [3]:
# ============================================================
# 3. PROJECT PATHS
# ============================================================

ROOT = "/content/drive/MyDrive/ENARES_2024_PROJECT"

RAW_DIR = os.path.join(ROOT, "01BasesDatosPrimarias")
LOG_DIR = os.path.join(ROOT, "05Resultados/logs")
REPORT_DIR = os.path.join(ROOT, "04CuestionariosInformes/reportes")

os.makedirs(LOG_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("ROOT:", ROOT)
print("RAW_DIR:", RAW_DIR)
print("LOG_DIR:", LOG_DIR)
print("REPORT_DIR:", REPORT_DIR)


ROOT: /content/drive/MyDrive/ENARES_2024_PROJECT
RAW_DIR: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias
LOG_DIR: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs
REPORT_DIR: /content/drive/MyDrive/ENARES_2024_PROJECT/04CuestionariosInformes/reportes


## Locate CRS04 raw `.sav` files

The expected CRS04 modules are:

- `976-Modulo1959`
- `976-Modulo1960`
- `976-Modulo1961`
- `976-Modulo1962`

The notebook searches recursively inside `01BasesDatosPrimarias` for `.sav` files containing `CRS04` in the filename.


In [4]:
# ============================================================
# 4. FIND CRS04 SAV FILES
# ============================================================

crs04_sav_paths = sorted(
    glob.glob(os.path.join(RAW_DIR, "**", "*CRS04*.sav"), recursive=True)
)

print(f"CRS04 .sav files found: {len(crs04_sav_paths)}")

for path in crs04_sav_paths:
    print("-", path)

EXPECTED_CRS04_FILES = 4

assert len(crs04_sav_paths) == EXPECTED_CRS04_FILES, (
    f"Expected {EXPECTED_CRS04_FILES} CRS04 .sav files, found {len(crs04_sav_paths)}"
)


CRS04 .sav files found: 4
- /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1959/extracted/976-Modulo1959/19_CRS04_CAP100.sav
- /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1960/extracted/976-Modulo1960/20_CRS04_CAP200.sav
- /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1961/extracted/976-Modulo1961/21_CRS04_CAP248.sav
- /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1962/extracted/976-Modulo1962/22_CRS04_CAP300.sav


## Read CRS04 files

To avoid memory problems in Colab, this notebook reads each CRS04 file separately and generates one raw profiling report per file.

This is safer than merging the four files, because Stage 1 does not perform merges.


In [5]:
# ============================================================
# 5. READ CRS04 FILES AND GENERATE RAW PROFILES
# ============================================================

profile_records = []

for sav_path in crs04_sav_paths:
    print("=" * 80)
    print("Reading:", sav_path)

    df, meta = pyreadstat.read_sav(sav_path)

    file_name = os.path.basename(sav_path)
    module_guess = "unknown"

    for part in sav_path.split(os.sep):
        if part.startswith("976-Modulo"):
            module_guess = part
            break

    print("Module:", module_guess)
    print("Rows:", len(df))
    print("Columns:", len(df.columns))

    profile_title = f"ENARES 2024 CRS04 Stage 1 Raw Profiling - {module_guess} - {file_name}"

    profile = ProfileReport(
        df,
        title=profile_title,
        explorative=True,
        minimal=True
    )

    html_name = file_name.replace(".sav", "_raw_profile.html")
    html_path = os.path.join(REPORT_DIR, html_name)

    profile.to_file(html_path)

    profile_records.append({
        "module_id": module_guess,
        "sav_file": file_name,
        "sav_path": sav_path,
        "n_rows": int(len(df)),
        "n_columns": int(len(df.columns)),
        "profile_html_path": html_path,
        "generated_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
        "privacy_note": "HTML profile remains in Google Drive and must not be pushed to GitHub."
    })

    print("Profile saved:", html_path)

print("=" * 80)
print("All CRS04 raw profiles generated.")


Reading: /content/drive/MyDrive/ENARES_2024_PROJECT/01BasesDatosPrimarias/Modulo1959/extracted/976-Modulo1959/19_CRS04_CAP100.sav
Module: 976-Modulo1959
Rows: 18807
Columns: 147


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 147/147 [00:02<00:00, 62.48it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

...


100%|██████████| 51/51 [00:00<00:00, 70.76it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Profile saved: /content/drive/MyDrive/ENARES_2024_PROJECT/04CuestionariosInformes/reportes/22_CRS04_CAP300_raw_profile.html
All CRS04 raw profiles generated.


In [6]:
# ============================================================
# 6. SAVE PROFILING INDEX
# ============================================================

profile_index_df = pd.DataFrame(profile_records)

profile_index_path = os.path.join(
    LOG_DIR,
    "ENARES_2024_CRS04_STAGE1_raw_profile_index.csv"
)

profile_index_df.to_csv(profile_index_path, index=False)

print("Profile index saved:", profile_index_path)
display(profile_index_df)


Profile index saved: /content/drive/MyDrive/ENARES_2024_PROJECT/05Resultados/logs/ENARES_2024_CRS04_STAGE1_raw_profile_index.csv


,module_id,sav_file,sav_path,n_rows,n_columns,profile_html_path,generated_at,privacy_note
0,976-Modulo1959,19_CRS04_CAP100.sav,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,147,/content/drive/MyDrive/ENARES_2024_PROJECT/04C...,2026-06-18 22:23:14,HTML profile remains in Google Drive and must ...
1,976-Modulo1960,20_CRS04_CAP200.sav,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,523,/content/drive/MyDrive/ENARES_2024_PROJECT/04C...,2026-06-18 22:25:06,HTML profile remains in Google Drive and must ...
2,976-Modulo1961,21_CRS04_CAP248.sav,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,578,/content/drive/MyDrive/ENARES_2024_PROJECT/04C...,2026-06-18 22:27:05,HTML profile remains in Google Drive and must ...
3,976-Modulo1962,22_CRS04_CAP300.sav,/content/drive/MyDrive/ENARES_2024_PROJECT/01B...,18807,51,/content/drive/MyDrive/ENARES_2024_PROJECT/04C...,2026-06-18 22:27:21,HTML profile remains in Google Drive and must ...


## Acceptance checks

This section verifies that the profiling notebook produced the expected outputs without altering raw files.


In [7]:
# ============================================================
# 7. ACCEPTANCE CHECKS
# ============================================================

assert len(profile_index_df) == EXPECTED_CRS04_FILES, (
    f"Expected {EXPECTED_CRS04_FILES} profiling records, found {len(profile_index_df)}"
)

assert profile_index_df["profile_html_path"].apply(os.path.exists).all(), (
    "At least one profiling HTML output was not found."
)

assert profile_index_df["n_rows"].gt(0).all(), (
    "At least one CRS04 file has zero rows."
)

assert profile_index_df["n_columns"].gt(0).all(), (
    "At least one CRS04 file has zero columns."
)

print("PASS: Notebook 5 profiling acceptance checks completed successfully.")


PASS: Notebook 5 profiling acceptance checks completed successfully.


## Addendum text for Stage 1 report

You can paste this text into the final Stage 1 report if needed.


In [8]:
# ============================================================
# 8. ADDENDUM TEXT FOR REPORT
# ============================================================

addendum_text = '''
## Raw CRS04 Profiling Addendum

A separate Stage 1 profiling notebook was created to generate raw exploratory reports for the CRS04 `.sav` files using `ydata-profiling`. The purpose of this profiling step is descriptive and technical: it documents missingness patterns, variable distributions, alerts, and structural characteristics of the raw CRS04 files.

This profiling does not clean, recode, merge, model, or infer. It remains within the Stage 1 scope of ingestion, metadata preservation, and structural validation.

The generated HTML profiling reports are stored in Google Drive under `04CuestionariosInformes/reportes`. Because these files may contain detailed raw-data distributions, rare values, or sensitive metadata patterns, they must not be pushed to GitHub. Only the notebook and sanitized documentation should be versioned.
'''

print(addendum_text)



## Raw CRS04 Profiling Addendum

A separate Stage 1 profiling notebook was created to generate raw exploratory reports for the CRS04 `.sav` files using `ydata-profiling`. The purpose of this profiling step is descriptive and technical: it documents missingness patterns, variable distributions, alerts, and structural characteristics of the raw CRS04 files.

This profiling does not clean, recode, merge, model, or infer. It remains within the Stage 1 scope of ingestion, metadata preservation, and structural validation.

The generated HTML profiling reports are stored in Google Drive under `04CuestionariosInformes/reportes`. Because these files may contain detailed raw-data distributions, rare values, or sensitive metadata patterns, they must not be pushed to GitHub. Only the notebook and sanitized documentation should be versioned.



## Final output expected

This notebook should generate:

- `04CuestionariosInformes/reportes/*_raw_profile.html`
- `05Resultados/logs/ENARES_2024_CRS04_STAGE1_raw_profile_index.csv`

GitHub should include:

- `notebooks/01_ingesta/05_ENARES_2024_STAGE1_perfilamiento.ipynb`

GitHub should **not** include:

- `.html` profiling reports
- `.sav` files
- `.zip` files
- credentials
- raw CSV/JSON outputs with private Drive IDs
